In [ ]:
import os
import sys

%load_ext autoreload
%autoreload 2
import logging
# Configure logging
logging.basicConfig(level=logging.INFO)

# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
import pickle
from server_config import datapath, preprocessed_path_freezed, redcap_path, preprocessed_path
from functions.preprocessing.ema_mappings import clean_heart_rate_data


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt



In [ ]:
from functions.preprocessing import gps_features
from functions.preprocessing.ema_mappings import run_ema_mappings
from functions.preprocessing.missing_data import summarize_missing_data

In [ ]:
import warnings
# Suppress only SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [ ]:
# backup_path = preprocessed_path_freezed + "/backup_data_passive_actual.feather"
backup_path = "/sc-projects/sc-proj-cc15-preact/SP6/raw/backup_passive_recent.feather" # new file 
df_backup = pd.read_feather(backup_path)
df_backup

In [ ]:
customers_with_100k_additional_measured_HR_seconds = ['N3CY', 'OmAV', 'DRtE', 'EtmK', 'G2Gk', 'lAHE', '4MLe', 'ZAjp', 'tilE', '0xWn', 'JIhV', 'WF9K', 'AfEo', 'g6p5', 'iIYT']
if False:
    df_backup = df_backup[~df_backup["customer"].isin(customers_with_100k_additional_measured_HR_seconds)]
df_backup

In [ ]:
df_backup["customer"].nunique() 

In [ ]:
df_backup.type.unique()

In [ ]:
df_backup["duration"] = (df_backup["endTimestamp"] - df_backup["startTimestamp"]).dt.total_seconds()
# 1 sec min and 24h max, mostly 60sec

df_backup["duration"].describe([0.01, 0.05, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])

In [ ]:
df_backup["customer"].nunique()

In [ ]:
# Convert the result to a DataFrame and print as markdown table
result = df_backup.groupby("type")["customer"].nunique().sort_values(ascending=False)
result_df = result.reset_index()
result_df.columns = ['Data Type', 'Unique Customers']
# print(result_df.to_markdown(index=False))
result

sums like below don't work, as the events overlap, so we overcount

In [ ]:
df_backup[df_backup["type"] == "ActiveBurnedCalories"].groupby(
    ["customer", "startTimestamp_day"]
)["doubleValue"].sum().sort_values(ascending=False)

In [ ]:
target_date = pd.to_datetime("2024-05-05")
target_date = pd.to_datetime("2023-12-11")
# df_backup[(df_backup["customer"] == "1BAf") & (df_backup["startTimestamp_day"] == target_date)]
df_backup[(df_backup["customer"] == "1BAf") & (df_backup["startTimestamp_day"] == target_date) & (df_backup["type"] == "ActiveBurnedCalories")]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["type"] == "ActiveBurnedCalories")]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["startTimestamp_day"] == target_date)]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["type"] == "ActiveBurnedCalories")]
# first record says that 17k calories were burned on that day something is off with this 1BAf customer

## HeartRate

to get rid of overlapping events, we need to "expand" the dataset, so that one row last only one second (the smalles time duration)

In [ ]:
# df = df_backup[(df_backup["type"] == "HeartRate") & (df_backup["customer"].isin(["4MLe","kVhY"]))]
df = df_backup[(df_backup["type"] == "HeartRate")]
df = df[["customer", "startTimestamp", "endTimestamp", "longValue"]]
df = df.rename(columns={"longValue": "HeartRate"})


In [ ]:
df["duration"] = (df["endTimestamp"] - df["startTimestamp"]).dt.total_seconds().fillna(0).astype(int)
df["duration"].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
df_expanded = df.loc[df.index.repeat(df['duration'])].copy()
df_expanded

In [ ]:
df_expanded['time_offset'] = df_expanded.groupby(level=0).cumcount()
df_expanded['timestamp'] = df_expanded['startTimestamp'] + pd.to_timedelta(df_expanded['time_offset'], unit='s')

as we can see, there are over 1M overlapping entries

In [ ]:
# df_expanded_groupby = df_expanded.groupby(['customer', 'timestamp'])
df_expanded_groupby = df_expanded.groupby(['timestamp', 'customer'])

df_expanded_groupby.size().value_counts().sort_index()

In [ ]:
df_size = df_expanded_groupby.size().reset_index(name='n_repeat')
# drop all the rows where count is 1
df_size = df_size[df_size['n_repeat'] > 1]
df_size.sort_values(by='n_repeat', ascending=False)


In [ ]:
df_size["n_repeat"].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])

In [ ]:
df_size["n_additional"] = df_size["n_repeat"] - 1
df_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
customer_additional = df_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
# Get all customers from df_backup and ensure they're included with 0 if not in customer_additional
all_customers = df_backup["customer"].unique()
customer_additional = customer_additional.reindex(all_customers, fill_value=0).sort_values(ascending=False)

print(customer_additional.describe([]).round(2))
customer_additional.quantile([0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99, 1.0], interpolation="linear").round(2)

by setting threshold of n_additional = 100_000 sec, we keep 97% of patients 

In [ ]:
plt.figure(figsize=(12, 8))
plt.bar(range(len(customer_additional)), customer_additional.values)

# Create percentage-based x-axis labels (reversed from 100% to 0%)
n_customers = len(customer_additional)
percentage_ticks = np.arange(0, n_customers, max(1, n_customers // 20))  # Show ~20 ticks
percentage_labels = [f"{100 - (i/n_customers)*100:.0f}%" for i in percentage_ticks]

plt.xticks(percentage_ticks, percentage_labels)
plt.xlabel('Patients Percentile (100% = highest duplicates, 0% = lowest duplicates)')
plt.ylabel('Number of Additional Duplicate Heart Rate Measurements (seconds)')
plt.title('Additional Duplicate Heart Rate Measurements by Customer')
plt.grid(True, alpha=0.3)
# plt.yscale('log')  # Use logarithmic scale for better visibility

# Add horizontal line at 100,000 additional measurements
plt.axhline(y=100000, color='red', linestyle='--', linewidth=2, label='100,000 threshold')

# Find the position where customer_additional crosses 100,000
threshold_index = (customer_additional > 100000).sum()
threshold_percentage = 100 - (threshold_index / n_customers) * 100

# Add vertical line at the threshold position
plt.axvline(x=threshold_index, color='red', linestyle='--', linewidth=2, alpha=0.7)

# Add text annotation
plt.text(threshold_index + 20, 100000 * 1.5, f'{threshold_percentage:.1f}% of patients\nhave <100k duplicates each', 
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.legend()
plt.tight_layout()

print(f"Threshold analysis:")
print(f"Number of customers with >100,000 additional duplicates: {threshold_index}")
print(f"Percentage of customers with >100,000 additional duplicates: {threshold_percentage:.1f}%")
print(f"Percentage of customers with ≤100,000 additional duplicates: {100 - threshold_percentage:.1f}%")

# Also show top 20 customers with most duplicates
print("\nTop 20 customers with most additional duplicates:")
customer_additional_top20 = customer_additional.head(20)
print(customer_additional_top20)

plt.xlim(-1, len(customer_additional)/2)  # Set x-axis limits to cover all customers
plt.show()


# Print customer names with >100,000 additional duplicates for easy copying
customers_over_threshold = customer_additional[customer_additional > 100000].index.tolist()
print(f"Customers with >100,000 additional duplicates ({len(customers_over_threshold)} total):")
print(customers_over_threshold)

In [ ]:
# Remove duplicates by taking the average where there are multiple entries per timestamp

df_avg = df_expanded_groupby.agg({
    'HeartRate': 'mean',
}).reset_index()

df_min = df_expanded_groupby.agg({
    'HeartRate': 'min',
}).reset_index()

df_max = df_expanded_groupby.agg({
    'HeartRate': 'max',
}).reset_index()

df_avg

In [ ]:
diff = df_max["HeartRate"] - df_min["HeartRate"]
print(f"proportion of repeated data that have different values: {(diff>0).sum() / (len(df_expanded) - len(df_avg)):.2%}")
print("out of this, the following are the statistics of the differences:")
diff[diff > 0].describe([0.25,0.5,0.75,0.9, 0.95, 0.99])
# TODO resolve what to do with this next, about ~80 of repeated 

In [ ]:
# df = df_avg[df_avg["customer"].isin(["1BAf", "lAHE", "4MLe", "kVhY"])]
df = df_avg

In [ ]:
df["HR_zone_resting"] = df["HeartRate"] < 60
df["HR_zone_moderate"] = (60 <= df["HeartRate"]) & (df["HeartRate"] < 100)
df["HR_zone_vigorous"] = (100 <= df["HeartRate"])

In [ ]:
df

In [ ]:
# Create hourly heart rate averages
df['hour'] = df['timestamp'].dt.floor('h')
df_hourly = df.groupby(['customer', 'hour']).agg(
    HeartRate_count=('HeartRate', 'count'),
    HeartRate_mean=('HeartRate', 'mean'),
    HeartRate_std=('HeartRate', 'std'),
    HeartRate_min=('HeartRate', 'min'),
    HeartRate_max=('HeartRate', 'max'),
    HeartRate_skew=('HeartRate', "skew"),
    HeartRate_kurtosis=('HeartRate', lambda x: x.kurtosis()),
    HeartRate_zone_resting=('HR_zone_resting', 'sum'),
    HeartRate_zone_moderate=('HR_zone_moderate', 'sum'),
    HeartRate_zone_vigorous=('HR_zone_vigorous', 'sum'),
).reset_index()

df_hourly

In [ ]:
# Create daily heart rate averages
df['day'] = df['timestamp'].dt.floor('d')
df_daily = df.groupby(['customer', 'day']).agg(
    HeartRate_count=('HeartRate', 'count'),
    HeartRate_mean=('HeartRate', 'mean'),
    HeartRate_std=('HeartRate', 'std'),
    HeartRate_min=('HeartRate', 'min'),
    HeartRate_max=('HeartRate', 'max'),
    HeartRate_skew=('HeartRate', "skew"),
    HeartRate_kurtosis=('HeartRate', lambda x: x.kurtosis()),
    HeartRate_zone_resting=('HR_zone_resting', 'sum'),
    HeartRate_zone_moderate=('HR_zone_moderate', 'sum'),
    HeartRate_zone_vigorous=('HR_zone_vigorous', 'sum'),
).reset_index()

df_daily

In [ ]:
df_plot = df_avg[(df_avg["customer"] == "05kz") & (df_avg["timestamp"] >= pd.to_datetime("2023-10-19")) 
                 & (df_avg["timestamp"] < pd.to_datetime("2023-10-22"))]
df_plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.scatter(df_plot['timestamp'], df_plot['HeartRate'], linewidth=0.5) # scatter vs line plot
plt.title(f'Heart Rate Data for Customer {df_plot["customer"].iloc[0]} on {df_plot["timestamp"].dt.date.iloc[0]} onwards')
plt.xlabel('Time')
plt.ylabel('Heart Rate (BPM)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Steps

In [ ]:
# Similar to HeartRate analysis, expand the Steps dataset
df_steps = df_backup[(df_backup["type"] == "Steps")]
df_steps = df_steps[["customer", "startTimestamp", "endTimestamp", "doubleValue"]]
df_steps = df_steps.rename(columns={"doubleValue": "Steps"})
df_steps

In [ ]:
print(f"{(df_steps["startTimestamp"].dt.second != 0).sum() / len(df_steps):.2%}")

In [ ]:
bins = np.arange(0, 61, 1) - 0.5  # 0 to 59 seconds
df_steps["startTimestamp"].dt.second.describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])
plt.figure(figsize=(10, 6))
plt.hist(df_steps["startTimestamp"].dt.second, bins=bins, alpha=0.5, edgecolor='black')
plt.hist(df_steps["endTimestamp"].dt.second, bins=bins, alpha=0.5, edgecolor='black')
plt.title('Distribution of Start Timestamp Seconds for Steps Data')
plt.xlabel('Second (0-59)')
plt.ylabel('Frequency')
plt.yscale('log')  # Use logarithmic scale for better visibility
plt.grid(True, alpha=0.3)
plt.xticks(np.arange(0, 60, 5))
plt.tight_layout()
plt.show()

such a samll number of samples got different offset than 0 (less than 2%), and 99% of samples durations are withing (59,61) range -- more than 90% is exactly 60s

therefore, just round the seconds 

In [ ]:
df_steps["duration"] = (df_steps["endTimestamp"] - df_steps["startTimestamp"]).dt.total_seconds().fillna(0).astype(int)
df_steps["duration"].describe([0.003, 0.004, 0.005, 0.0075, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.993, 0.994, 0.995, 0.9975, 0.999])

In [ ]:
mask_under_59 = df_steps["duration"] < 59
mask_over_61 = df_steps["duration"] > 61
print(f"Percentage of samples with duration not in [59, 61]: {((mask_under_59 | mask_over_61).sum() / len(df_steps)) * 100:.2f}%")
df_steps["StepsPerMinute"] = df_steps["Steps"] / (df_steps["duration"] / 60)
df_steps["StepsPerMinute"].describe([0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99, 0.995])

In [ ]:
df_steps["StepsPerMinute"][mask_under_59].describe([0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99, 0.995])

In [ ]:
df_steps["StepsPerMinute"][mask_over_61].describe([0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99, 0.995])

In [ ]:
df_steps["startTimestamp_minute"] = df_steps["startTimestamp"].dt.round('min')
# (df_steps["duration"] / 60).round(0).astype
df_steps["duration_minutes"] = np.maximum(1, (df_steps["duration"] / 60).round(0).astype(int))
df_steps["duration_minutes"].min()

In [ ]:
df_steps_expanded = df_steps.loc[df_steps.index.repeat(df_steps['duration_minutes'])].copy()
df_steps_expanded

In [ ]:
df_steps_expanded['time_offset'] = df_steps_expanded.groupby(level=0).cumcount()
df_steps_expanded['timestamp'] = df_steps_expanded['startTimestamp'] + pd.to_timedelta(df_steps_expanded['time_offset'], unit='min')
df_steps_expanded

In [ ]:
# Check for overlapping entries in Steps data
df_steps_expanded_groupby = df_steps_expanded.groupby(['timestamp', 'customer'])

df_steps_expanded_groupby.size().value_counts().sort_index()

In [ ]:
df_steps_size = df_steps_expanded_groupby.size().reset_index(name='n_repeat')
# drop all the rows where count is 1
df_steps_size = df_steps_size[df_steps_size['n_repeat'] > 1]
df_steps_size.sort_values(by='n_repeat', ascending=False)

In [ ]:
df_steps_size["n_repeat"].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])

In [ ]:
df_steps_size["n_additional"] = df_steps_size["n_repeat"] - 1
df_steps_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
customer_steps_additional = df_steps_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
# Get all customers from df_backup and ensure they're included with 0 if not in customer_steps_additional
all_customers = df_backup["customer"].unique()
customer_steps_additional = customer_steps_additional.reindex(all_customers, fill_value=0).sort_values(ascending=False)

print(customer_steps_additional.describe([]).round(2))
customer_steps_additional.quantile([0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99, 1.0], interpolation="linear").round(2)

In [ ]:
plt.figure(figsize=(12, 8))
plt.bar(range(len(customer_steps_additional)), customer_steps_additional.values)

# Create percentage-based x-axis labels (reversed from 100% to 0%)
n_customers = len(customer_steps_additional)
percentage_ticks = np.arange(0, n_customers, max(1, n_customers // 20))  # Show ~20 ticks
percentage_labels = [f"{100 - (i/n_customers)*100:.0f}%" for i in percentage_ticks]

plt.xticks(percentage_ticks, percentage_labels)
plt.xlabel('Patients Percentile (100% = highest duplicates, 0% = lowest duplicates)')
plt.ylabel('Number of Additional Duplicate Steps Measurements (seconds)')
plt.title('Additional Duplicate Steps Measurements by Customer')
plt.grid(True, alpha=0.3)

# Add horizontal line at 100,000 additional measurements
# plt.axhline(y=100000, color='red', linestyle='--', linewidth=2, label='100,000 threshold')

# Find the position where customer_steps_additional crosses 100,000
threshold_index = (customer_steps_additional > 100000).sum()
threshold_percentage = 100 - (threshold_index / n_customers) * 100

# Add vertical line at the threshold position
# plt.axvline(x=threshold_index, color='red', linestyle='--', linewidth=2, alpha=0.7)

# Add text annotation
# plt.text(threshold_index + 20, 100000 * 1.5, f'{threshold_percentage:.1f}% of patients\nhave <100k duplicates each', 
        #  bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.legend()
plt.tight_layout()

print(f"Steps Threshold analysis:")
print(f"Number of customers with >100,000 additional duplicates: {threshold_index}")
print(f"Percentage of customers with >100,000 additional duplicates: {threshold_percentage:.1f}%")
print(f"Percentage of customers with ≤100,000 additional duplicates: {100 - threshold_percentage:.1f}%")

# Also show top 20 customers with most duplicates
print("\nTop 20 customers with most additional Steps duplicates:")
customer_steps_additional_top20 = customer_steps_additional.head(20)
print(customer_steps_additional_top20)

plt.xlim(-1, len(customer_steps_additional)/2)  # Set x-axis limits to cover all customers
plt.show()

# Print customer names with >100,000 additional duplicates for easy copying
customers_steps_over_threshold = customer_steps_additional[customer_steps_additional > 100000].index.tolist()
print(f"Customers with >100,000 additional Steps duplicates ({len(customers_steps_over_threshold)} total):")
print(customers_steps_over_threshold)

In [ ]:
# Remove duplicates by taking the average where there are multiple entries per timestamp

df_steps_avg = df_steps_expanded_groupby.agg({
    'StepsPerMinute': 'mean',
}).reset_index()

df_steps_min = df_steps_expanded_groupby.agg({
    'StepsPerMinute': 'min',
}).reset_index()

df_steps_max = df_steps_expanded_groupby.agg({
    'StepsPerMinute': 'max',
}).reset_index()

df_steps_avg

In [ ]:
diff_steps = df_steps_max["StepsPerMinute"] - df_steps_min["StepsPerMinute"]
print(f"Proportion of repeated Steps data that have different values: {(diff_steps>0).sum() / (len(df_steps_expanded) - len(df_steps_avg)):.2%}")
print("Out of this, the following are the statistics of the differences:")
diff_steps[diff_steps > 0].describe([0.25,0.5,0.75,0.9, 0.95, 0.99])

In [ ]:
df_steps_final = df_steps_avg
df_steps_final

In [ ]:
# Create hourly steps aggregates
df_steps_final['hour'] = df_steps_final['timestamp'].dt.floor('h')
df_steps_hourly = df_steps_final.groupby(['customer', 'hour']).agg(
    StepsPerMinute_count=('StepsPerMinute', 'count'),
    StepsPerHour=('StepsPerMinute', 'sum'),
    StepsPerMinute_mean=('StepsPerMinute', 'mean'),
    StepsPerMinute_std=('StepsPerMinute', 'std'),
    StepsPerMinute_min=('StepsPerMinute', 'min'),
    StepsPerMinute_max=('StepsPerMinute', 'max'),
    StepsPerMinute_skew=('StepsPerMinute', "skew"),
    StepsPerMinute_kurtosis=('StepsPerMinute', lambda x: x.kurtosis()),
).reset_index()

df_steps_hourly

In [ ]:
# Create daily steps aggregates
df_steps_final['day'] = df_steps_final['timestamp'].dt.floor('d')
df_steps_daily = df_steps_final.groupby(['customer', 'day']).agg(
    StepsPerMinute_count=('StepsPerMinute', 'count'),
    StepsPerDay=('StepsPerMinute', 'sum'),
    StepsPerMinute_mean=('StepsPerMinute', 'mean'),
    StepsPerMinute_std=('StepsPerMinute', 'std'),
    StepsPerMinute_min=('StepsPerMinute', 'min'),
    StepsPerMinute_max=('StepsPerMinute', 'max'),
    StepsPerMinute_skew=('StepsPerMinute', "skew"),
    StepsPerMinute_kurtosis=('StepsPerMinute', lambda x: x.kurtosis()),
).reset_index()

df_steps_daily

In [ ]:
df_steps_plot = df_steps_final[(df_steps_final["customer"] == "05kz") & (df_steps_final["timestamp"] >= pd.to_datetime("2023-10-19")) 
                 & (df_steps_final["timestamp"] < pd.to_datetime("2023-10-22"))]
df_steps_plot

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(df_steps_plot['timestamp'], df_steps_plot['StepsPerMinute'], linewidth=0.5) # scatter vs line plot
plt.title(f'Steps Data for Customer {df_steps_plot["customer"].iloc[0]} on {df_steps_plot["timestamp"].dt.date.iloc[0]} onwards')
plt.xlabel('Time')
plt.ylabel('Steps per Second')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()